In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

csv_path = next(Path("../data/raw").glob("*.csv"))
print("Loading:", csv_path)

df = pd.read_csv(csv_path)
print(df.shape)
df.head()

Loading: ..\data\raw\zameen-property-data.csv
(168446, 17)


,property_id,location_id,page_url,property_type,price,location,city,province_name,latitude,longitude,baths,area,purpose,bedrooms,date_added,agency,agent
0,237062,3325,https://www.zameen.com/Property/g_10_g_10_2_gr...,Flat,10000000,G-10,Islamabad,Islamabad Capital,33.679890,73.012640,2,4 Marla,For Sale,2,02-04-2019,NaN,NaN
1,346905,3236,https://www.zameen.com/Property/e_11_2_service...,Flat,6900000,E-11,Islamabad,Islamabad Capital,33.700993,72.971492,3,5.6 Marla,For Sale,3,05-04-2019,NaN,NaN
2,386513,764,https://www.zameen.com/Property/islamabad_g_15...,House,16500000,G-15,Islamabad,Islamabad Capital,33.631486,72.926559,6,8 Marla,For Sale,5,07-17-2019,NaN,NaN
3,656161,340,https://www.zameen.com/Property/islamabad_bani...,House,43500000,Bani Gala,Islamabad,Islamabad Capital,33.707573,73.151199,4,2 Kanal,For Sale,4,04-05-2019,NaN,NaN
4,841645,3226,https://www.zameen.com/Property/dha_valley_dha...,House,7000000,DHA Defence,Islamabad,Islamabad Capital,33.492591,73.301339,3,8 Marla,For Sale,3,07-10-2019,Easy Property,Muhammad Junaid Ceo Muhammad Shahid Director


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 168446 entries, 0 to 168445
Data columns (total 17 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   property_id    168446 non-null  int64  
 1   location_id    168446 non-null  int64  
 2   page_url       168446 non-null  str    
 3   property_type  168446 non-null  str    
 4   price          168446 non-null  int64  
 5   location       168446 non-null  str    
 6   city           168446 non-null  str    
 7   province_name  168446 non-null  str    
 8   latitude       168446 non-null  float64
 9   longitude      168446 non-null  float64
 10  baths          168446 non-null  int64  
 11  area           168446 non-null  str    
 12  purpose        168446 non-null  str    
 13  bedrooms       168446 non-null  int64  
 14  date_added     168446 non-null  str    
 15  agency         124375 non-null  str    
 16  agent          124374 non-null  str    
dtypes: float64(2), int64(5), str(10)
memory 

In [4]:
df.isnull().sum()

property_id          0
location_id          0
page_url             0
property_type        0
price                0
location             0
city                 0
province_name        0
latitude             0
longitude            0
baths                0
area                 0
purpose              0
bedrooms             0
date_added           0
agency           44071
agent            44072
dtype: int64

In [5]:
for col in ["property_type", "purpose", "city"]:
    if col in df.columns:
        print(df[col].value_counts(), "\n")

property_type
House            105468
Flat              38238
Upper Portion     13774
Lower Portion      9229
Room                685
Farm House          657
Penthouse           395
Name: count, dtype: int64 

purpose
For Sale    120655
For Rent     47791
Name: count, dtype: int64 

city
Karachi       60484
Lahore        41427
Islamabad     37426
Rawalpindi    20976
Faisalabad     8133
Name: count, dtype: int64 



In [6]:
print(df["area"].sample(15, random_state=1).tolist())
print(df["area"].str.extract(r"([A-Za-z].*)")[0].value_counts())

['1 Kanal', '3.8 Marla', '5 Marla', '4.9 Marla', '14 Marla', '7 Marla', '4 Marla', '8 Marla', '5 Marla', '10 Marla', '9.3 Marla', '5.8 Marla', '10 Marla', '4.7 Marla', '1 Kanal']
0
Marla    137769
Kanal     30677
Name: count, dtype: int64


In [7]:
houses = df[(df["purpose"] == "For Sale") & (df["property_type"] == "House")].copy()
print(houses.shape)

(87976, 17)


In [8]:
import re

def area_to_marla(s):
    s = str(s).replace(",", "").strip()
    m = re.match(r"([\d.]+)\s*(.*)", s)
    if not m:
        return np.nan
    val, unit = float(m.group(1)), m.group(2).lower()
    if "kanal" in unit:
        return val * 20
    if "marla" in unit:
        return val
    if "yd" in unit or "yard" in unit:
        return val * 9 / 225
    if "ft" in unit or "feet" in unit:
        return val / 225
    return np.nan

houses["area_marla"] = houses["area"].apply(area_to_marla)
print("Could not parse:", houses["area_marla"].isna().sum())
houses[["area", "area_marla"]].sample(10, random_state=1)

Could not parse: 0


,area,area_marla
6842,1 Kanal,20.0
9819,5 Marla,5.0
10840,3.7 Marla,3.7
139236,1 Kanal,20.0
131912,7 Marla,7.0
122778,8 Marla,8.0
122905,11 Marla,11.0
164956,10 Marla,10.0
105809,10 Marla,10.0
167285,5 Marla,5.0


In [9]:
before = len(houses)

houses = houses[(houses["price"] > 0) & (houses["area_marla"] > 0)]
houses = houses[(houses["bedrooms"] > 0) & (houses["bedrooms"] <= 15)]

houses["price_per_marla"] = houses["price"] / houses["area_marla"]
low, high = houses["price_per_marla"].quantile([0.01, 0.99])
houses = houses[houses["price_per_marla"].between(low, high)]

print(f"Removed {before - len(houses)} rows, {len(houses)} left")
houses[["price", "area_marla", "bedrooms", "baths", "price_per_marla"]].describe()

Removed 11517 rows, 76459 left


,price,area_marla,bedrooms,baths,price_per_marla
count,7.645900e+04,76459.000000,76459.000000,76459.000000,7.645900e+04
mean,2.894658e+07,10.801923,4.317386,3.956643,2.429195e+06
std,3.966094e+07,9.862695,1.535744,2.633213,1.148906e+06
min,1.200000e+06,1.000000,1.000000,0.000000,6.250000e+05
25%,1.000000e+07,5.000000,3.000000,3.000000,1.708333e+06
50%,1.650000e+07,8.000000,4.000000,4.000000,2.200000e+06
75%,3.250000e+07,13.000000,5.000000,6.000000,2.800000e+06
max,2.000000e+09,384.000000,15.000000,403.000000,7.916667e+06


In [11]:
print("baths == 0:", (houses["baths"] == 0).sum())
print("area > 100 Marla:", (houses["area_marla"] > 100).sum())

before = len(houses)
houses = houses[(houses["baths"] >= 1) & (houses["baths"] <= 15)]
houses = houses[houses["area_marla"] <= 100]   # up to 5 Kanal

print(f"Removed {before - len(houses)} more, {len(houses)} left")
houses[["price", "area_marla", "bedrooms", "baths"]].describe().round(1)

baths == 0: 10262
area > 100 Marla: 49
Removed 10301 more, 66158 left


,price,area_marla,bedrooms,baths
count,66158.0,66158.0,66158.0,66158.0
mean,28327355.5,10.7,4.3,4.6
std,36136765.9,8.9,1.5,1.7
min,1200000.0,1.0,1.0,1.0
25%,10000000.0,5.0,3.0,3.0
50%,16500000.0,8.0,4.0,4.0
75%,32500000.0,14.0,5.0,6.0
max,750000000.0,100.0,13.0,13.0


In [10]:
keep = ["price", "area_marla", "bedrooms", "baths", "city",
        "location", "latitude", "longitude", "date_added"]
houses = houses[keep]

from pathlib import Path
Path("../data/processed").mkdir(exist_ok=True)
houses.to_csv("../data/processed/houses_clean.csv", index=False)
print("Saved:", houses.shape)

Saved: (76459, 9)
